In [1]:
import os

print("Current Directory:")
print(os.getcwd())

print("\nFiles and Folders:")
for item in os.listdir():
    print(item)

Current Directory:
c:\Users\ASUS\Videos\EE200_course_project_data_2026\EE200_Q3

Files and Folders:
app3b.py
app4.py
Bohemian Rhapsody.mp3
fingerprint_cache.pkl
Hey Jude.mp3
plot_01_full_dft.png
plot_02_window_comparison.png
plot_03_noise_sweep.png
plot_query_constellation.png
plot_query_offset_histogram.png
plot_query_offset_histogram_singlepeak.png
plot_query_spectrogram.png
Q3A.ipynb
query.wav
run_log.txt
songs


In [2]:
import os
import sys
import time
import librosa
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt

from scipy.signal import spectrogram
from scipy.ndimage import maximum_filter
from collections import defaultdict, Counter

# Force stdout to flush immediately so progress is visible as it
# happens rather than appearing in one burst at the end (or never,
# if the process is killed before exit). Also tee everything to a
# log file, since notebook output panels (e.g. VSCode) truncate long
# output and you'll want the full numbers for your report anyway.
_log_file = open("run_log.txt", "a", buffering=1)

def print(*args, **kwargs):
    text = " ".join(str(a) for a in args)
    __builtins__.print(text, **{**kwargs, "flush": True})
    _log_file.write(text + "\n")


# =====================================================
# PARAMETERS
# =====================================================

SONG_FOLDER = "songs"
QUERY_FILE = "query.mp3"

SR = 22050
NPERSEG = 2048
NOVERLAP = 1024

PEAK_PERCENTILE = 99
NEIGHBORHOOD_SIZE = 20
FAN_VALUE = 10


# =====================================================
# BASIC AUDIO / SPECTROGRAM HELPERS
# =====================================================

_AUDIO_CACHE = {}

def load_audio(path, duration=None):
    cache_key = (path, duration)
    if cache_key in _AUDIO_CACHE:
        return _AUDIO_CACHE[cache_key]

    t0 = time.perf_counter()
    # Try librosa's modern soxr_hq resampler (fast, usually bundled
    # already); fall back to whatever librosa's plain default is if
    # neither soxr nor resampy is installed in this environment.
    try:
        audio, sr = librosa.load(path, sr=SR, mono=True, duration=duration, res_type="soxr_hq")
    except (ModuleNotFoundError, ImportError):
        # Neither soxr nor resampy available — fall back to librosa's
        # built-in default resampler (slower, but always available).
        audio, sr = librosa.load(path, sr=SR, mono=True, duration=duration)
    elapsed = time.perf_counter() - t0
    if elapsed > 2:
        print(f"  (loaded {os.path.basename(path)} in {elapsed:.1f}s)")

    _AUDIO_CACHE[cache_key] = (audio, sr)
    return audio, sr


def compute_spectrogram(audio, sr, nperseg=NPERSEG, noverlap=NOVERLAP):
    f, t, Sxx = spectrogram(
        audio,
        fs=sr,
        nperseg=nperseg,
        noverlap=noverlap
    )
    Sxx_db = 10 * np.log10(Sxx + 1e-10)
    return f, t, Sxx_db


# =====================================================
# PART 1 — WHY A SINGLE DFT IS NOT ENOUGH
#
# A plain DFT of the whole song tells you which frequencies are
# present overall, but flattens the entire duration into one
# spectrum: all information about *when* each frequency occurred
# is destroyed. We plot this once, then immediately show the
# spectrogram alternative which keeps time on one axis.
# =====================================================

def plot_full_dft(audio_path):

    audio, sr = load_audio(audio_path)

    spectrum = np.fft.rfft(audio)
    freqs = np.fft.rfftfreq(len(audio), d=1 / sr)
    magnitude = np.abs(spectrum)

    plt.figure(figsize=(10, 4))
    plt.plot(freqs, magnitude)
    plt.title("Whole-Song DFT Magnitude (no timing information)")
    plt.xlabel("Frequency (Hz)")
    plt.ylabel("Magnitude")
    plt.xlim(0, 5000)
    plt.tight_layout()
    plt.savefig("plot_01_full_dft.png", dpi=120)
    plt.close()

    print("Saved: plot_01_full_dft.png")
    print(
        "Observation: the DFT shows which frequencies are present in the "
        "whole clip, but every time-axis is collapsed into this single "
        "spectrum. Two completely different orderings of the same notes "
        "would produce an identical plot, so this cannot be used to "
        "fingerprint *when* something happens in the song."
    )


# =====================================================
# PART 2 — SHORT VS LONG WINDOW SPECTROGRAM
#
# The window length sets the time-frequency resolution tradeoff:
# a short window localises events precisely in time but smears
# frequency content (wide bins); a long window resolves frequency
# finely but blurs together events that are close in time.
# =====================================================

def compare_window_lengths(audio_path, short_win=512, long_win=4096):

    audio, sr = load_audio(audio_path)

    f_short, t_short, Sxx_short = compute_spectrogram(
        audio, sr, nperseg=short_win, noverlap=short_win // 2
    )
    f_long, t_long, Sxx_long = compute_spectrogram(
        audio, sr, nperseg=long_win, noverlap=long_win // 2
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].pcolormesh(t_short, f_short, Sxx_short, shading="gouraud")
    axes[0].set_title(f"Short window (nperseg={short_win})\nGood time resolution, poor frequency resolution")
    axes[0].set_xlabel("Time (s)")
    axes[0].set_ylabel("Frequency (Hz)")

    axes[1].pcolormesh(t_long, f_long, Sxx_long, shading="gouraud")
    axes[1].set_title(f"Long window (nperseg={long_win})\nGood frequency resolution, poor time resolution")
    axes[1].set_xlabel("Time (s)")
    axes[1].set_ylabel("Frequency (Hz)")

    plt.tight_layout()
    plt.savefig("plot_02_window_comparison.png", dpi=120)
    plt.close()

    print("Saved: plot_02_window_comparison.png")
    print(
        "Observation: with the short window, fast events (note onsets, "
        "transients) are sharply localised in time but each frequency bin "
        "is wide, so the spectrogram looks blurry/streaky vertically. With "
        "the long window, frequency bins are narrow and tones appear as "
        "thin, well-separated horizontal lines, but the window now spans "
        "a much longer stretch of audio, so fast changes get smeared "
        "together horizontally. This is the standard time-frequency "
        "resolution tradeoff (uncertainty principle for windowed Fourier "
        "transforms): you cannot have arbitrarily fine resolution in both "
        "domains at once, NPERSEG=2048 is a compromise that gives "
        "adequate resolution in both for fingerprinting purposes."
    )


# =====================================================
# PEAK PICKING
# =====================================================

def find_peaks(Sxx):

    local_max = (
        Sxx ==
        maximum_filter(Sxx, size=NEIGHBORHOOD_SIZE)
    )

    threshold = np.percentile(Sxx, PEAK_PERCENTILE)

    detected = local_max & (Sxx > threshold)

    peaks = np.argwhere(detected)

    return peaks


# =====================================================
# PAIRED HASHES (original approach)
# =====================================================

def create_hashes(peaks, freqs, times):

    hashes = []

    peaks = sorted(peaks, key=lambda x: x[1])

    for i in range(len(peaks)):
        for j in range(1, FAN_VALUE + 1):

            if i + j >= len(peaks):
                break

            p1 = peaks[i]
            p2 = peaks[i + j]

            f1 = int(freqs[p1[0]])
            f2 = int(freqs[p2[0]])

            t1 = times[p1[1]]
            t2 = times[p2[1]]

            dt = round(t2 - t1, 2)

            if dt > 0:
                hash_key = (f1, f2, dt)
                hashes.append((hash_key, t1))

    return hashes


# =====================================================
# PART 3 — SINGLE-PEAK "HASHES" (for comparison)
#
# Instead of pairing two peaks, each peak on its own becomes a
# fingerprint: just (frequency, time). This is far less specific,
# since many songs share isolated frequency bins, so it produces
# many more coincidental matches.
# =====================================================

def create_single_peak_hashes(peaks, freqs, times):

    hashes = []

    for p in peaks:
        f1 = int(freqs[p[0]])
        t1 = times[p[1]]
        hash_key = (f1,)
        hashes.append((hash_key, t1))

    return hashes


# =====================================================
# DATABASE BUILDING (parametrised by which hashing function to use)
# =====================================================

def build_database(song_folder, hash_fn=create_hashes, file_list=None):

    database = defaultdict(list)

    print(f"Building fingerprint database using {hash_fn.__name__}...\n")

    files = file_list if file_list is not None else os.listdir(song_folder)
    files = [f for f in files if f.lower().endswith((".mp3", ".wav"))]
    total = len(files)

    for idx, file in enumerate(files, 1):

        t0 = time.perf_counter()

        print("Processing:", file)

        path = os.path.join(song_folder, file)

        audio, sr = load_audio(path)
        f, t, Sxx = compute_spectrogram(audio, sr)
        peaks = find_peaks(Sxx)
        hashes = hash_fn(peaks, f, t)

        for h, offset in hashes:
            database[h].append((file, offset))

        print(f"  [{idx}/{total}] {file} done in {time.perf_counter()-t0:.1f}s "
              f"({len(hashes)} hashes)")

    print("\nDatabase Built Successfully!\n")
    return database


# =====================================================
# IDENTIFICATION (parametrised the same way)
# =====================================================

def identify_song(query_file, database, hash_fn=create_hashes):

    audio, sr = load_audio(query_file)
    f, t, Sxx = compute_spectrogram(audio, sr)
    peaks = find_peaks(Sxx)
    hashes = hash_fn(peaks, f, t)

    votes = defaultdict(list)

    for h, query_time in hashes:

        if h not in database:
            continue

        for song_name, db_time in database[h]:
            offset = round(db_time - query_time, 1)
            votes[song_name].append(offset)

    best_song = None
    best_score = 0
    best_offsets = None

    for song, offsets in votes.items():
        count = Counter(offsets)
        score = max(count.values())

        if score > best_score:
            best_score = score
            best_song = song
            best_offsets = offsets

    return best_song, best_score, best_offsets


def identify_song_from_audio(audio, sr, database, hash_fn=create_hashes):
    """Same as identify_song but takes an in-memory audio array
    instead of a file path, so noise/pitch/stretch experiments don't
    need to write a temp file for every trial."""

    f, t, Sxx = compute_spectrogram(audio, sr)
    peaks = find_peaks(Sxx)
    hashes = hash_fn(peaks, f, t)

    votes = defaultdict(list)

    for h, query_time in hashes:

        if h not in database:
            continue

        for song_name, db_time in database[h]:
            offset = round(db_time - query_time, 1)
            votes[song_name].append(offset)

    best_song = None
    best_score = 0
    best_offsets = None

    for song, offsets in votes.items():
        count = Counter(offsets)
        score = max(count.values())

        if score > best_score:
            best_score = score
            best_song = song
            best_offsets = offsets

    return best_song, best_score, best_offsets


# =====================================================
# PLOTTING HELPERS
# =====================================================

def plot_spectrogram(audio_path, save_as="plot_spectrogram.png"):

    audio, sr = load_audio(audio_path)
    f, t, Sxx = compute_spectrogram(audio, sr)

    plt.figure(figsize=(10, 6))
    plt.pcolormesh(t, f, Sxx, shading="gouraud")
    plt.title("Spectrogram")
    plt.xlabel("Time (s)")
    plt.ylabel("Frequency (Hz)")
    plt.colorbar(label="dB")
    plt.tight_layout()
    plt.savefig(save_as, dpi=120)
    plt.close()
    print(f"Saved: {save_as}")


def plot_constellation(audio_path, save_as="plot_constellation.png"):

    audio, sr = load_audio(audio_path)
    f, t, Sxx = compute_spectrogram(audio, sr)
    peaks = find_peaks(Sxx)

    plt.figure(figsize=(10, 6))
    plt.scatter(t[peaks[:, 1]], f[peaks[:, 0]], s=5)
    plt.title("Constellation Map")
    plt.xlabel("Time (s)")
    plt.ylabel("Frequency (Hz)")
    plt.tight_layout()
    plt.savefig(save_as, dpi=120)
    plt.close()
    print(f"Saved: {save_as}")


def plot_offset_histogram(offsets, save_as="plot_offset_histogram.png", title="Offset Histogram"):

    plt.figure(figsize=(10, 6))
    plt.hist(offsets, bins=50)
    plt.title(title)
    plt.xlabel("Offset")
    plt.ylabel("Votes")
    plt.tight_layout()
    plt.savefig(save_as, dpi=120)
    plt.close()
    print(f"Saved: {save_as}")


# =====================================================
# AUDIO DISTORTION HELPERS
# =====================================================

def add_noise(audio, sigma):
    noise = np.random.normal(0, sigma, len(audio))
    return audio + noise


# =====================================================
# PART 4 — NOISE ROBUSTNESS SWEEP
#
# Add increasing amounts of Gaussian noise to the query and find the
# point at which the matcher stops recognising the correct song (or
# stops finding any match above the trivial minimum score).
# =====================================================

def noise_robustness_sweep(query_path, true_song_name, database,
                            sigmas=(0.0, 0.02, 0.05, 0.1, 0.2, 0.4)):

    # Use a short clip (<=10s) for these repeated experiments rather
    # than the full track — each sweep step reprocesses the audio
    # from scratch, so working on a clip instead of a multi-minute
    # song keeps each step to a fraction of a second instead of tens
    # of seconds.
    audio, sr = load_audio(query_path, duration=10)

    print("\n--- Noise Robustness Sweep ---")
    results = []

    for sigma in sigmas:

        noisy = add_noise(audio, sigma) if sigma > 0 else audio
        song, score, offsets = identify_song_from_audio(noisy, sr, database)

        correct = (song == true_song_name)
        results.append((sigma, song, score, correct))

        print(f"sigma={sigma:>5} -> predicted={song!s:<25} score={score:<5} correct={correct}")

    # Plot score vs sigma to visualise the breakdown point
    sig_vals = [r[0] for r in results]
    scores = [r[2] for r in results]

    plt.figure(figsize=(8, 5))
    plt.plot(sig_vals, scores, marker="o")
    plt.title("Match Score vs Noise Level")
    plt.xlabel("Noise sigma")
    plt.ylabel("Best match score")
    plt.tight_layout()
    plt.savefig("plot_03_noise_sweep.png", dpi=120)
    plt.close()
    print("Saved: plot_03_noise_sweep.png")

    first_fail = next((r for r in results if not r[3]), None)
    if first_fail:
        print(
            f"Observation: recognition first fails at sigma={first_fail[0]}. "
            "As noise increases, the spectrogram peaks that survive the "
            "PEAK_PERCENTILE threshold are increasingly noise peaks rather "
            "than true tonal content, so fewer of the query's hashes exist "
            "in the database at all, and of those that do, fewer line up "
            "at a single consistent offset. The match score decays "
            "gradually rather than failing all at once, because the "
            "fingerprint is built from many redundant local hashes."
        )
    else:
        print(
            "Observation: the matcher survived all tested noise levels; "
            "try larger sigma values to find the actual breakdown point."
        )

    return results


# =====================================================
# PART 5 — PITCH SHIFT / TIME STRETCH ROBUSTNESS
#
# Unlike noise, pitch shifting and time stretching are *structured*
# distortions: they systematically move every peak's frequency or
# time coordinate. Since hashes encode exact (f1, f2, dt) triples,
# even a tiny shift moves peaks into different frequency bins / time
# bins and the hash no longer matches anything in the database,
# even though a human still recognises the song instantly.
# =====================================================

def pitch_time_robustness_test(query_path, true_song_name, database):

    audio, sr = load_audio(query_path, duration=10)

    print("\n--- Pitch Shift / Time Stretch Robustness ---")

    pitch_shift_steps = [0, 0.25, 0.5, 1.0]
    for n_steps in pitch_shift_steps:
        shifted = librosa.effects.pitch_shift(audio, sr=sr, n_steps=n_steps) if n_steps else audio
        song, score, _ = identify_song_from_audio(shifted, sr, database)
        print(f"pitch_shift={n_steps:>5} semitones -> predicted={song!s:<25} "
              f"score={score:<5} correct={song == true_song_name}")

    time_stretch_rates = [1.00, 1.02, 1.05, 1.10]
    for rate in time_stretch_rates:
        stretched = librosa.effects.time_stretch(audio, rate=rate) if rate != 1.0 else audio
        song, score, _ = identify_song_from_audio(stretched, sr, database)
        print(f"time_stretch_rate={rate:>5} -> predicted={song!s:<25} "
              f"score={score:<5} correct={song == true_song_name}")

    print(
        "Observation: even a fraction-of-a-semitone pitch shift, or a "
        "few-percent time stretch, can collapse the match score to "
        "near zero. This is because the hash key is built from *exact* "
        "frequency bin values (f1, f2) and an *exact* time gap (dt). A "
        "pitch shift multiplies every frequency by a constant ratio, "
        "moving almost every peak into a different bin, so essentially "
        "none of the original (f1, f2, dt) triples survive intact in "
        "the new clip even though the harmonic *pattern* is unchanged. "
        "A time stretch similarly rescales every dt. To a human ear the "
        "song is unmistakably the same, because our perception is "
        "relative/log-frequency and tolerant of small tempo changes, "
        "but the exact-match hash lookup has zero tolerance. "
        "One fix: quantise frequencies into coarser log-spaced bins "
        "and/or store relative frequency ratios instead of absolute "
        "Hz values, and allow a small tolerance window when matching "
        "dt instead of requiring an exact bin match. This trades some "
        "specificity for robustness to small pitch/tempo perturbations."
    )


# =====================================================
# MAIN
# =====================================================

def list_songs(song_folder, limit=None):
    songs = sorted(
        f for f in os.listdir(song_folder)
        if f.lower().endswith((".mp3", ".wav"))
    )
    if not songs:
        raise FileNotFoundError(f"No .mp3/.wav files found in '{song_folder}/'")
    if limit is not None:
        songs = songs[:limit]
    return songs


if __name__ == "__main__":

    np.random.seed(42)  # comment out / change for a different random pick each run

    # Set this to a small number (e.g. 5) while developing/debugging to
    # avoid reprocessing all 50 songs every run; set to None for the
    # real full run.
    SUBSET_SIZE = None

    all_songs = list_songs(SONG_FOLDER, limit=SUBSET_SIZE)
    print(f"Found {len(all_songs)} songs in '{SONG_FOLDER}/' (using {len(all_songs)}).")

    # Pick one random song to use for the DFT/window demos, and a
    # (possibly different) random song to use as the query for
    # identification + noise/pitch/stretch experiments.
    demo_song_name = np.random.choice(all_songs)
    TRUE_SONG = np.random.choice(all_songs)

    print(f"Using '{demo_song_name}' for DFT/window demos.")
    print(f"Using '{TRUE_SONG}' as the query song for identification experiments.")

    # ---- Part 1: single DFT loses timing information ----
    sample_song = os.path.join(SONG_FOLDER, demo_song_name)
    plot_full_dft(sample_song)

    # ---- Part 2: short vs long window spectrogram ----
    compare_window_lengths(sample_song)

    # ---- Build a query clip from the chosen demo song (10s clip) ----
    audio, sr = librosa.load(sample_song, sr=22050, mono=True)
    clip_len = 10 * sr
    start = 0 if len(audio) <= clip_len else np.random.randint(0, len(audio) - clip_len)
    end = start + clip_len
    query_audio = audio[start:end]
    sf.write("query.wav", query_audio, sr)

    # ---- Build paired-hash database (the main approach) ----
    database = build_database(SONG_FOLDER, hash_fn=create_hashes, file_list=all_songs)

    QUERY_FILE = os.path.join(SONG_FOLDER, TRUE_SONG)

    # Use a short, randomly-chosen clip from the query song rather than
    # the entire track — identifying the full original file against
    # itself is trivial (every hash matches, so the score is just the
    # song's total hash count) and isn't a real recognition test.
    q_audio, q_sr = load_audio(QUERY_FILE)
    clip_len_q = 10 * q_sr
    q_start = 0 if len(q_audio) <= clip_len_q else np.random.randint(0, len(q_audio) - clip_len_q)
    q_clip = q_audio[q_start:q_start + clip_len_q]
    sf.write("query_identify.wav", q_clip, q_sr)
    QUERY_CLIP_FILE = "query_identify.wav"
    print(f"Identification query is a {clip_len_q / q_sr:.0f}s clip starting at "
          f"{q_start / q_sr:.1f}s into '{TRUE_SONG}'.")

    song, score, offsets = identify_song(QUERY_CLIP_FILE, database, hash_fn=create_hashes)
    print("\n[Paired hashes] Predicted Song :", song)
    print("[Paired hashes] Match Score:", score)

    plot_spectrogram(QUERY_CLIP_FILE, "plot_query_spectrogram.png")
    plot_constellation(QUERY_CLIP_FILE, "plot_query_constellation.png")
    if offsets is not None:
        plot_offset_histogram(offsets, "plot_query_offset_histogram.png",
                               title="Offset Histogram (paired hashes)")

    # ---- Part 3: single-peak hashing, for comparison ----
    # Reuse the already-loaded audio cache instead of reprocessing all
    # 50 songs from scratch a second time.
    print("\n--- Single-Peak vs Paired-Hash Comparison ---")
    database_single = build_database(SONG_FOLDER, hash_fn=create_single_peak_hashes, file_list=all_songs)
    song_s, score_s, offsets_s = identify_song(
        QUERY_CLIP_FILE, database_single, hash_fn=create_single_peak_hashes
    )
    print("[Single peaks]  Predicted Song :", song_s)
    print("[Single peaks]  Match Score:", score_s)

    if offsets_s is not None:
        plot_offset_histogram(offsets_s, "plot_query_offset_histogram_singlepeak.png",
                               title="Offset Histogram (single peaks)")

    print(
        "Observation: single-peak hashes are far less specific than "
        "paired hashes, since a lone (frequency, time) pair is shared by "
        "huge numbers of unrelated notes across many songs. This produces "
        "a much higher rate of coincidental votes scattered across many "
        "candidate songs and offsets, so the winning offset's vote count "
        "is a much smaller, less decisive fraction of the total than with "
        "paired hashes. Pairing two peaks (and their time gap) creates a "
        "far rarer, more specific signature, so a true match concentrates "
        "votes sharply at one offset while a wrong song's votes stay "
        "low and spread out, exactly the contrast the identifier relies "
        "on to decide a match confidently."
    )

    # ---- Part 4: noise robustness sweep ----
    noise_robustness_sweep(QUERY_CLIP_FILE, TRUE_SONG, database)

    # ---- Part 5: pitch shift / time stretch robustness ----
    pitch_time_robustness_test(QUERY_CLIP_FILE, TRUE_SONG, database)

    print("\nAll experiments completed. See plot_*.png files for figures.")

Found 50 songs in 'songs/' (using 50).
Using 'Taxman.mp3' for DFT/window demos.
Using 'Lucy In The Sky With Diamonds.mp3' as the query song for identification experiments.
  (loaded Taxman.mp3 in 4.5s)
Saved: plot_01_full_dft.png
Observation: the DFT shows which frequencies are present in the whole clip, but every time-axis is collapsed into this single spectrum. Two completely different orderings of the same notes would produce an identical plot, so this cannot be used to fingerprint *when* something happens in the song.
Saved: plot_02_window_comparison.png
Observation: with the short window, fast events (note onsets, transients) are sharply localised in time but each frequency bin is wide, so the spectrogram looks blurry/streaky vertically. With the long window, frequency bins are narrow and tones appear as thin, well-separated horizontal lines, but the window now spans a much longer stretch of audio, so fast changes get smeared together horizontally. This is the standard time-freque